In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.layers import *


#connecting to google drive



In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!ls /content/drive/MyDrive/tiny-imagenet/

test  train  val


In [4]:
train_dir = '/content/drive/MyDrive/tiny-imagenet/train'
val_dir = '/content/drive/MyDrive/tiny-imagenet/val'
test_dir = '/content/drive/MyDrive/tiny-imagenet/test'

#laoding the data

In [5]:
img_size = (128,128)
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=img_size,
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=img_size,
    batch_size=batch_size
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=img_size,
    batch_size=batch_size
)

num_classes = len(train_ds.class_names)
print("Classes:", train_ds.class_names)


Found 3500 files belonging to 10 classes.
Found 500 files belonging to 10 classes.
Found 1000 files belonging to 10 classes.
Classes: ['n07871810', 'n07873807', 'n07875152', 'n07920052', 'n09193705', 'n09246464', 'n09256479', 'n09332890', 'n09428293', 'n12267677']


##creating model without augmentation

In [6]:

inputs2 = layers.Input(shape=(128,128,3))
x = layers.Rescaling(1./255)(inputs2)
# inputs2=layers.Input(shape=(128,128,3))
x=Conv2D(12,kernel_size=(3,3),padding="same",input_shape=(128,128,3))(inputs2)
x=BatchNormalization()(x)
x=Activation("relu")(x)
x=Conv2D(12,kernel_size=(3,3),padding="same")(x)
x=BatchNormalization()(x)
x=Activation("relu")(x)
x=MaxPool2D(pool_size=(2,2),strides=2,padding="same")(x)
x=Dropout(0.1)(x)

x=Conv2D(24,kernel_size=(3,3),padding="same")(x)
x=BatchNormalization()(x)
x=Activation("relu")(x)
x=Conv2D(24,kernel_size=(3,3),padding="same")(x)
x=BatchNormalization()(x)
x=Activation("relu")(x)
x=MaxPool2D(pool_size=(2,2),strides=2,padding="same")(x)
x=Dropout(0.25)(x)

x=Conv2D(48,kernel_size=(3,3),padding="same")(x)
x=BatchNormalization(name="layer5")(x)
x=Activation("relu")(x)
x=Conv2D(48,kernel_size=(3,3),padding="same")(x)
x=BatchNormalization()(x)
x=Activation("relu")(x)
x=MaxPool2D(pool_size=(2,2),strides=2,padding="same")(x)
x=Dropout(0.25)(x)
x=GlobalAveragePooling2D()(x)
x=Dense(32,activation="relu")(x)
x=BatchNormalization()(x)
x=Dropout(0.25)(x)
# x=Dense(32,activation="relu")(x)
# x=Dropout(0.4)(x)
outputs2=Dense(10,activation="softmax")(x)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
model1=Model(inputs=inputs2,outputs=outputs2)
model1.summary()



Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 12)   │           336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 12)   │            48 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 128, 128, 12)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 128, 12)   │         1,308 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128, 128, 12)   │            48 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 128, 128, 12)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 12)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64, 64, 12)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 24)     │         2,616 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64, 64, 24)     │            96 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 64, 64, 24)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 64, 64, 24)     │         5,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64, 64, 24)     │            96 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 64, 64, 24)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 24)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32, 32, 24)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 32, 32, 48)     │        10,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer5 (BatchNormalization)     │ (None, 32, 32, 48)     │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 32, 32, 48)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 32, 32, 48)     │        20,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 32, 32, 48)     │           192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 43,366 (169.40 KB)

 Trainable params: 42,966 (167.84 KB)

 Non-trainable params: 400 (1.56 KB)

In [10]:
import tensorflow as tf
print("GPU Available:", tf.config.list_physical_devices('GPU'))

GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


#Training the model without augmentaion

In [11]:
model1.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])
model1.fit(train_ds,validation_data=val_ds,epochs=50,callbacks=[keras.callbacks.EarlyStopping(patience=10)])

Epoch 1/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 767s 7s/step - accuracy: 0.3380 - loss: 1.7704 - val_accuracy: 0.2440 - val_loss: 2.0455
Epoch 2/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 12s 110ms/step - accuracy: 0.3994 - loss: 1.6221 - val_accuracy: 0.4420 - val_loss: 1.5905
Epoch 3/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 12s 107ms/step - accuracy: 0.4328 - loss: 1.5320 - val_accuracy: 0.4340 - val_loss: 1.4885
Epoch 4/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 12s 105ms/step - accuracy: 0.4612 - loss: 1.4314 - val_accuracy: 0.3760 - val_loss: 1.7014
Epoch 5/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 12s 107ms/step - accuracy: 0.4608 - loss: 1.4152 - val_accuracy: 0.4220 - val_loss: 1.6827
Epoch 6/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 12s 105ms/step - accuracy: 0.5114 - loss: 1.3269 - val_accuracy: 0.5080 - val_loss: 1.4735
Epoch 7/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 11s 104ms/step - accuracy: 0.5241 - loss: 1.2799 - val_accuracy: 0.5280 - val_loss: 1.3263
Epoch 8/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 21s 105ms/step - accuracy: 0.5589 - loss: 1.2

#Testing the model without augmentation

In [12]:
test_loss1, test_acc1 = model1.evaluate(test_ds)
training_loss1,training_acc1=model1.evaluate(train_ds)
print("Test Accuracy without augmentaion",test_acc1)
print("Training Accuracy without augmentaion",training_acc1)

32/32 ━━━━━━━━━━━━━━━━━━━━ 375s 12s/step - accuracy: 0.5895 - loss: 1.1993
110/110 ━━━━━━━━━━━━━━━━━━━━ 11s 99ms/step - accuracy: 0.6529 - loss: 0.9338
Test Accuracy without augmentaion 0.6079999804496765
Training Accuracy without augmentaion 0.6660000085830688


#Creating model with augmentation

In [13]:
data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.1),
    layers.RandomBrightness(0.2),
    layers.RandomContrast(0.2),
])

In [18]:
x = data_augmentation(inputs2)
x = layers.Rescaling(1./255)(x)

# inputs2=layers.Input(shape=(128,128,3))
x=Conv2D(12,kernel_size=(3,3),padding="same")(x)
x=BatchNormalization()(x)
x=Activation("relu")(x)
x=Conv2D(12,kernel_size=(3,3),padding="same")(x)
x=BatchNormalization()(x)
x=Activation("relu")(x)
x=MaxPool2D(pool_size=(2,2),strides=2,padding="same")(x)
x=Dropout(0.2)(x)

x=Conv2D(24,kernel_size=(3,3),padding="same")(x)
x=BatchNormalization()(x)
x=Activation("relu")(x)
x=Conv2D(24,kernel_size=(3,3),padding="same")(x)
x=BatchNormalization()(x)
x=Activation("relu")(x)
x=MaxPool2D(pool_size=(2,2),strides=2,padding="same")(x)
x=Dropout(0.3)(x)

x=Conv2D(48,kernel_size=(3,3),padding="same")(x)
x=BatchNormalization(name="layer5")(x)
x=Activation("relu")(x)
x=Conv2D(48,kernel_size=(3,3),padding="same")(x)
x=BatchNormalization()(x)
x=Activation("relu")(x)
x=MaxPool2D(pool_size=(2,2),strides=2,padding="same")(x)
x=Dropout(0.2)(x)
x=GlobalAveragePooling2D()(x)
x=Dense(32,activation="relu")(x)
x=BatchNormalization()(x)
x=Dropout(0.1)(x)
# x=Dense(32,activation="relu")(x)
# x=Dropout(0.4)(x)
outputs3=Dense(10,activation="softmax")(x)
# model2=Model(inputs=inputs2,outputs=outputs2)
# model2.summary()

In [19]:
model2=Model(inputs2,outputs3)
model2.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_2 (Rescaling)         │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 128, 128, 12)   │           336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 128, 128, 12)   │            48 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_12 (Activation)      │ (None, 128, 128, 12)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 128, 128, 12)   │         1,308 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 128, 128, 12)   │            48 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_13 (Activation)      │ (None, 128, 128, 12)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 64, 64, 12)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64, 64, 12)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 64, 64, 24)     │         2,616 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_14          │ (None, 64, 64, 24)     │            96 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_14 (Activation)      │ (None, 64, 64, 24)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 64, 64, 24)     │         5,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_15          │ (None, 64, 64, 24)     │            96 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_15 (Activation)      │ (None, 64, 64, 24)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 32, 32, 24)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 32, 32, 24)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_16 (Conv2D)              │ (None, 32, 32, 48)     │        10,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer5 (BatchNormalization)     │ (None, 32, 32, 48)     │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_16 (Activation)      │ (None, 32, 32, 48)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ (None, 32, 32, 48)     │        20,78

 Total params: 43,366 (169.40 KB)

 Trainable params: 42,966 (167.84 KB)

 Non-trainable params: 400 (1.56 KB)

#Training model with Augmentation

In [20]:
model2.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model2.fit(train_ds, validation_data=val_ds, epochs=50,callbacks=[keras.callbacks.EarlyStopping(patience=10)])

Epoch 1/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 18s 121ms/step - accuracy: 0.2973 - loss: 1.9578 - val_accuracy: 0.1040 - val_loss: 2.3604
Epoch 2/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 13s 117ms/step - accuracy: 0.3932 - loss: 1.6311 - val_accuracy: 0.1560 - val_loss: 2.4299
Epoch 3/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 13s 115ms/step - accuracy: 0.4299 - loss: 1.5456 - val_accuracy: 0.1780 - val_loss: 2.4756
Epoch 4/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 13s 115ms/step - accuracy: 0.4562 - loss: 1.4699 - val_accuracy: 0.3320 - val_loss: 1.7896
Epoch 5/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 21s 117ms/step - accuracy: 0.4589 - loss: 1.4713 - val_accuracy: 0.3520 - val_loss: 1.8766
Epoch 6/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 13s 117ms/step - accuracy: 0.4714 - loss: 1.4238 - val_accuracy: 0.4380 - val_loss: 1.5519
Epoch 7/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 13s 116ms/step - accuracy: 0.4828 - loss: 1.3928 - val_accuracy: 0.3520 - val_loss: 1.8582
Epoch 8/50
110/110 ━━━━━━━━━━━━━━━━━━━━ 13s 119ms/step - accuracy: 0.5188 - loss: 1

#Testing Model with Augmentation

In [21]:
training_loss2,training_acc2=model2.evaluate(train_ds)
test_loss2, test_acc2 = model2.evaluate(test_ds)
print("Test Accuracy with augmentaion",test_acc2)
print("Training Accuracy with augmentaion",training_acc2)

110/110 ━━━━━━━━━━━━━━━━━━━━ 10s 94ms/step - accuracy: 0.6639 - loss: 0.8758
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 84ms/step - accuracy: 0.6321 - loss: 1.0247
Test Accuracy with augmentaion 0.6309999823570251
Training Accuracy with augmentaion 0.6811428666114807
